# 第 1 章 · 描述性统计

**描述性统计** 用少量数字概括一堆数据的样子，回答三类问题：

1. **数据是什么类型？**（定类 / 定序 / 定距 / 定比）
2. **中心在哪？**（平均数、中位数、众数）
3. **分散程度如何？**（极差、方差、标准差、变异系数）

本章全部代码可直接运行，看结果即可。

## 0. 准备环境

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 让 matplotlib 正确显示中文与负号
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei"]
plt.rcParams["axes.unicode_minus"] = False

np.random.seed(42)   # 固定随机种子，保证每次运行结果一致
print("环境就绪：numpy", np.__version__, "| pandas", pd.__version__)

## 1. 数据的四种测量尺度

| 类型 | 特点 | 例子 |
|---|---|---|
| 定类 Nominal | 只是分类，不能比大小 | 性别、血型 |
| 定序 Ordinal | 可以排序，但差值无意义 | 满意度：差/中/好 |
| 定距 Interval | 差值有意义，零点任意 | 摄氏度 |
| 定比 Ratio | 零点绝对，可算倍数 | 身高、体重、成绩 |

只有**定距**和**定比**数据才适合算平均数，定类数据只能算众数。

In [ ]:
levels = pd.DataFrame({
    "类型": ["定类", "定序", "定距", "定比"],
    "能否排序": ["否", "能", "能", "能"],
    "差值有意义": ["否", "否", "能", "能"],
    "零点绝对": ["否", "否", "否", "能"],
    "例子": ["性别", "满意度", "摄氏温度", "身高"],
})
levels

## 2. 集中趋势：均值 / 中位数 / 众数

In [ ]:
scores = pd.Series([78, 85, 92, 67, 88, 90, 73, 85, 95, 60, 85, 71])

print("均值   =", scores.mean())
print("中位数 =", scores.median())
print("众数   =", scores.mode().tolist())      # 可能不止一个
print("最大值 =", scores.max(), " 最小值 =", scores.min())

**要点**：均值对极端值敏感；中位数稳健。比如工资数据常被少数高薪拉高均值，
这时中位数更能代表"普通人"。

In [ ]:
# 极端值对均值 vs 中位数的影响
normal = [10, 12, 13, 15, 16, 18, 20]
with_outlier = normal + [1000]
print("无极端值 -> 均值 %.2f, 中位数 %.2f" % (np.mean(normal), np.median(normal)))
print("有极端值 -> 均值 %.2f, 中位数 %.2f" % (np.mean(with_outlier), np.median(with_outlier)))

## 3. 离散程度：极差 / 方差 / 标准差 / 变异系数

In [ ]:
x = np.array([78, 85, 92, 67, 88, 90, 73, 85, 95, 60, 85, 71])

rng = x.max() - x.min()                       # 极差
var = x.var(ddof=1)                           # 样本方差（分母 n-1）
std = x.std(ddof=1)                           # 样本标准差
cv = std / x.mean() * 100                     # 变异系数（%）

print("极差     =", rng)
print("样本方差 = %.2f" % var)
print("样本标准差 = %.2f" % std)
print("变异系数 = %.2f%%" % cv)

**为什么样本方差要除以 n-1？** 用样本估计总体方差时，样本均值已经"用掉"了 1 个自由度，
除以 n-1 才是无偏估计（贝塞尔校正）。

**标准差**：与数据同单位，最好解释。
**变异系数**：无量纲，可比较不同量纲数据的离散程度。

## 4. 分位数、箱线图、五数概括

In [ ]:
print("五数概括:")
print("  最小值 Q0 =", x.min())
print("  下四分位 Q1 =", np.percentile(x, 25))
print("  中位数 Q2 =", np.percentile(x, 50))
print("  上四分位 Q3 =", np.percentile(x, 75))
print("  最大值 Q4 =", x.max())
print("四分位距 IQR = Q3 - Q1 =", np.percentile(x, 75) - np.percentile(x, 25))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].boxplot(x, vert=True, patch_artist=True,
                boxprops=dict(facecolor="#cfe3ff"))
axes[0].set_title("箱线图（看分布与异常值）")
axes[0].set_ylabel("分数")

axes[1].hist(x, bins=6, color="#6da8f5", edgecolor="white")
axes[1].axvline(x.mean(), color="red", linestyle="--", label="均值")
axes[1].axvline(np.median(x), color="green", linestyle=":", label="中位数")
axes[1].set_title("直方图")
axes[1].legend()

plt.tight_layout()
plt.show()

## 5. 偏度与峰度：分布长什么样

In [ ]:
def skewness(a):
    a = np.asarray(a, dtype=float)
    n = len(a)
    m = a.mean()
    s = a.std(ddof=1)
    return n / ((n - 1) * (n - 2)) * np.sum(((a - m) / s) ** 3)

def kurtosis(a):
    a = np.asarray(a, dtype=float)
    n = len(a)
    m = a.mean()
    s = a.std(ddof=1)
    return (n * (n + 1) / ((n - 1) * (n - 2) * (n - 3)) *
            np.sum(((a - m) / s) ** 4) - 3 * (n - 1) ** 2 / ((n - 2) * (n - 3)))

print("偏度 = %.3f  (>0 右偏/正偏，<0 左偏/负偏，≈0 对称)" % skewness(x))
print("峰度 = %.3f  (>0 尖峰厚尾，<0 平峰，≈0 接近正态)" % kurtosis(x))

## 小结

- **中心**：均值（敏感）/ 中位数（稳健）/ 众数（分类数据）
- **离散**：极差、方差、标准差、变异系数、IQR
- **形状**：偏度、峰度、直方图、箱线图

**练习**：把 `scores` 换成你自己课程的成绩，看看均值和中位数差别多大。